# IntuBote: training a depth-0 chess transformer on Colab

IntuBote picks a move from the position alone, with no search. It learns from Stockfish evaluations in the Lichess evaluation database.

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Run the cells from top to bottom. Data and checkpoints go to Google Drive.
3. **If Colab disconnects,** run all the cells again. The data cell skips work that's already done, and training resumes from the last checkpoint.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/juandanielsg/chess-bote.git"
BRANCH   = "main"                         # the branch that contains intubote/

DRIVE_DIR = "/content/drive/MyDrive/intubote"
POSITIONS = 5_000_000                     # Lichess positions to use (~90 bytes each in RAM)
RUN_NAME  = "intubote_small"

# Model size. "small" ~7M params is a good first run on a T4; "medium" ~26M needs more data/steps.
SIZE = "small"
D_MODEL, LAYERS, HEADS = {"tiny": (128, 4, 4), "small": (256, 8, 8), "medium": (512, 8, 8)}[SIZE]

STEPS  = 100_000                          # optimizer steps (x BATCH positions each)
BATCH  = 1024
LR     = 3e-4
POLICY_TARGET = "best"                    # "best" = copy Stockfish's move, "soft" = spread over top candidates

## 2. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime -> Change runtime type"

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

%cd /content
!rm -rf chess-bote
!git clone -q --branch {BRANCH} {REPO_URL} chess-bote
%cd /content/chess-bote
# IntuBote only needs chess-bote's pure-Python board (for legal moves), so no C++ build is needed.
os.environ["PYTHONPATH"] = "/content/chess-bote/src"
!pip install -q zstandard orjson
!test -f intubote/train.py && echo "intubote/ found" || echo "ERROR: branch has no intubote/ (push it first)"

## 3. Data: stream positions from the Lichess evaluation database

This reads only the first `POSITIONS` entries of the multi-GB file and never downloads the whole thing. It runs at roughly 10k positions/s, so 5M takes about 10 minutes. The output is saved to Drive and reused in later sessions.

In [ ]:
DATA_DIR = f"{DRIVE_DIR}/data_{POSITIONS // 1_000_000}M"
if os.path.exists(f"{DATA_DIR}/meta.json"):
    print("data already prepared:", open(f"{DATA_DIR}/meta.json").read())
else:
    !python intubote/prepare_data.py --count {POSITIONS} --out /content/data_tmp
    !mkdir -p {DATA_DIR} && cp /content/data_tmp/* {DATA_DIR}/

# Train from local disk; Drive is slow to read.
!mkdir -p /content/data && cp -n {DATA_DIR}/* /content/data/

## 4. Train

Every 1,000 steps it prints validation **top-1** (how often it picks Stockfish's move) and **top-3**. Higher is better; compare runs by these numbers, but judge strength by playing games. Checkpoints are saved to Drive every 2,000 steps, and rerunning this cell resumes from the last one.

In [ ]:
CKPT = f"{DRIVE_DIR}/{RUN_NAME}.pt"
!python intubote/train.py --data /content/data --out {CKPT}     --steps {STEPS} --batch {BATCH} --lr {LR} --policy-target {POLICY_TARGET}     --d-model {D_MODEL} --layers {LAYERS} --heads {HEADS}

## 5. Try it

In [ ]:
for fen in ["rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
            "r1bqkb1r/pppp1ppp/2n2n2/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR w KQkq - 4 4",   # Qxf7#
            "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5Q2/PPPP1PPP/RNB1K1NR b KQkq - 3 3"]:  # stop Qxf7#
    print(fen)
    !python intubote/engine.py --ckpt {CKPT} --fen "{fen}" --top 3

## 6. Play it on your machine

Download `RUN_NAME.pt` from your Drive folder, then in the repo:

```bash
uv sync --group intubote
uv run python intubote/uci.py --ckpt path/to/intubote_small.pt     # add this command as a UCI engine in CuteChess/Arena
```

To measure strength, play it against chess-bote or Stockfish at limited Elo with `cutechess-cli`, like `scripts/match.sh` does.